# Notebook 02: Save & Load Models — Solutions

**Module**: Production & Deployment  
**Level**: Beginner

Solutions for all exercises in Notebook 02.

---
## Setup

In [1]:
import json

# Helpers
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from panelbox import load_model
from panelbox.core.results import PanelResults
from panelbox.gmm import DifferenceGMM
from panelbox.gmm.results import GMMResults
from panelbox.models.static.fixed_effects import FixedEffects

# PanelBox imports
from panelbox.models.static.pooled_ols import PooledOLS

sys.path.insert(0, str(Path("..")))
from utils.visualization_helpers import set_production_style

# Configuration
np.random.seed(42)
warnings.filterwarnings("ignore")
set_production_style()

# Paths
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "outputs" / "models"
FIGURES_DIR = BASE_DIR / "outputs" / "figures"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Setup complete.")

Setup complete.


In [2]:
# Load datasets
df_firms = pd.read_csv(DATA_DIR / "firm_panel.csv")
df_new_firms = pd.read_csv(DATA_DIR / "new_firms.csv")
df_lgd = pd.read_csv(DATA_DIR / "bank_lgd.csv")
df_new_bank = pd.read_csv(DATA_DIR / "new_bank_data.csv")

print(f"Firm panel:    {df_firms.shape[0]:,} rows")
print(f"New firms:     {df_new_firms.shape[0]:,} rows")
print(f"Bank LGD:      {df_lgd.shape[0]:,} rows")
print(f"New bank data: {df_new_bank.shape[0]:,} rows")

Firm panel:    2,000 rows
New firms:     100 rows
Bank LGD:      3,000 rows
New bank data: 150 rows


In [3]:
# Estimate models needed for exercises
model_ols = PooledOLS(
    "investment ~ value + capital + sales", df_firms, entity_col="firm_id", time_col="year"
)
results_ols = model_ols.fit()

model_gmm = DifferenceGMM(
    data=df_lgd,
    dep_var="lgd_logit",
    lags=1,
    exog_vars=["saldo_real", "pib_growth", "selic", "collateral_ratio"],
    id_var="contract_id",
    time_var="month",
    collapse=True,
    time_dummies=False,
)
results_gmm = model_gmm.fit()

print("PooledOLS and GMM models estimated.")
print(f"OLS R-squared: {results_ols.rsquared:.4f}")
print(f"GMM Hansen J p-value: {results_gmm.hansen_j.pvalue:.4f}")

PooledOLS and GMM models estimated.
OLS R-squared: 0.3273
GMM Hansen J p-value: 0.1219


---
## Exercise 1 (Easy): Save & Load a Fixed Effects Model

1. Estimate a `FixedEffects` model on `df_firms` with `entity_effects=True`
2. Save it to `outputs/models/fe_model.pkl`
3. Load it back using `load_model()`
4. Verify that entity fixed effects are preserved

In [4]:
# Step 1: Estimate Fixed Effects model
model_fe = FixedEffects(
    "investment ~ value + capital + sales",
    df_firms,
    entity_col="firm_id",
    time_col="year",
    entity_effects=True,
)
results_fe = model_fe.fit()
print(results_fe.summary())

                       Fixed Effects Estimation Results                       
Formula: investment ~ value + capital + sales
Model:   Fixed Effects
------------------------------------------------------------------------------
No. Observations:               2,000
No. Entities:                     100
No. Time Periods:                  20
Degrees of Freedom:             1,897
R-squared:                     0.5216
Adj. R-squared:                0.4959
R-squared (within):            0.5216
R-squared (between):           1.0000
R-squared (overall):           0.7702
Standard Errors:            nonrobust
F-statistic (FE vs OLS):      36.9275
F-test p-value:                0.0000
Variable        Coef.        Std.Err.     t        P>|t|    [0.025     0.975]    
------------------------------------------------------------------------------
value                0.2881      0.0079  36.424  0.0000    0.2726    0.3036 ***
capital              0.2106      0.0295   7.131  0.0000    0.1526    0.2685 

In [5]:
# Step 2: Save to pickle
fe_path = MODELS_DIR / "fe_model.pkl"
results_fe.save(str(fe_path))
print(f"Saved to: {fe_path}")
print(f"File size: {fe_path.stat().st_size / 1024:.1f} KB")

Saved to: ../outputs/models/fe_model.pkl
File size: 269.7 KB


In [6]:
# Step 3: Load using universal loader
loaded_fe = load_model(str(fe_path))
print(f"Loaded type: {type(loaded_fe).__name__}")
print(f"Model: {loaded_fe.model_type}")

Loaded type: PanelResults
Model: Fixed Effects


In [7]:
# Step 4: Verify entity fixed effects are preserved
# The FE are stored on the results object as _entity_fe
original_fe = results_fe._entity_fe
loaded_entity_fe = loaded_fe._entity_fe

print(f"Original entity FE: {len(original_fe)} entities")
print(f"Loaded entity FE:   {len(loaded_entity_fe)} entities")
print()

# Compare first 10 entity FE
comparison = pd.DataFrame(
    {
        "Original FE": original_fe.head(10),
        "Loaded FE": loaded_entity_fe.head(10),
        "Difference": (original_fe - loaded_entity_fe).head(10),
    }
)
print(comparison.to_string())
print()

assert np.allclose(original_fe.values, loaded_entity_fe.values), "FE mismatch!"
print("All entity fixed effects preserved exactly.")

Original entity FE: 100 entities
Loaded entity FE:   100 entities

    Original FE  Loaded FE  Difference
1      0.646565   0.646565         0.0
2      0.381264   0.381264         0.0
3      0.731178   0.731178         0.0
4      1.032810   1.032810         0.0
5     -0.390786  -0.390786         0.0
6     -0.421909  -0.421909         0.0
7      1.521384   1.521384         0.0
8      0.755208   0.755208         0.0
9     -0.342955  -0.342955         0.0
10     0.661763   0.661763         0.0

All entity fixed effects preserved exactly.


In [8]:
# Bonus: verify predictions on known entities use the preserved FE
train_ids = set(df_firms["firm_id"].unique())
known = df_new_firms[df_new_firms["firm_id"].isin(train_ids)].copy()

preds_orig = results_fe.predict(known)
preds_loaded = loaded_fe.predict(known)

print("Predictions on known entities:")
print(f"  Max difference: {np.max(np.abs(preds_orig - preds_loaded)):.2e}")
assert np.allclose(preds_orig, preds_loaded), "Prediction mismatch!"
print("  Predictions match exactly (FE are being used correctly).")

Predictions on known entities:
  Max difference: 0.00e+00
  Predictions match exactly (FE are being used correctly).


---
## Exercise 2 (Medium): Save Isolation Test

Demonstrate that saving creates an independent copy.

In [9]:
# Step 1: Re-estimate OLS to get a clean object
model_ols2 = PooledOLS(
    "investment ~ value + capital + sales", df_firms, entity_col="firm_id", time_col="year"
)
results_ols2 = model_ols2.fit()

# Save the original coefficients for reference
original_params = results_ols2.params.copy()
print("Original coefficients:")
print(original_params)
print()

Original coefficients:
Intercept   -0.507405
value        0.299045
capital      0.277294
sales        0.167421
dtype: float64



In [10]:
# Step 2: Save to pickle
isolation_path = MODELS_DIR / "isolation_test.pkl"
results_ols2.save(str(isolation_path))
print(f"Saved to: {isolation_path}")

Saved to: ../outputs/models/isolation_test.pkl


In [11]:
# Step 3: Modify the original object (set all coefficients to 0)
results_ols2.params[:] = 0.0

print("Modified coefficients (all zeroed out):")
print(results_ols2.params)
print()

Modified coefficients (all zeroed out):
Intercept    0.0
value        0.0
capital      0.0
sales        0.0
dtype: float64



In [12]:
# Step 4: Load the saved file — should have ORIGINAL values
loaded_isolation = load_model(str(isolation_path))

print("Loaded coefficients (from saved file):")
print(loaded_isolation.params)
print()

# Verify loaded matches original (pre-modification)
print("Comparison:")
comp = pd.DataFrame(
    {
        "Original (saved)": original_params,
        "Modified in-memory": results_ols2.params,
        "Loaded from file": loaded_isolation.params,
    }
)
print(comp.to_string())
print()

assert np.allclose(loaded_isolation.params.values, original_params.values), (
    "Loaded does not match original!"
)
assert not np.allclose(results_ols2.params.values, original_params.values), (
    "Modified should differ!"
)
print("CONFIRMED: save() creates an independent snapshot.")
print("Modifying the in-memory object does NOT affect the saved file.")

Loaded coefficients (from saved file):
Intercept   -0.507405
value        0.299045
capital      0.277294
sales        0.167421
dtype: float64

Comparison:
           Original (saved)  Modified in-memory  Loaded from file
Intercept         -0.507405                 0.0         -0.507405
value              0.299045                 0.0          0.299045
capital            0.277294                 0.0          0.277294
sales              0.167421                 0.0          0.167421

CONFIRMED: save() creates an independent snapshot.
Modifying the in-memory object does NOT affect the saved file.


---
## Exercise 3 (Medium): Export Model Card

Create a function that exports a model card as JSON.

In [13]:
def export_model_card(results, filepath):
    """
    Export a model card as a JSON file.

    Works with both PanelResults and GMMResults.

    Parameters
    ----------
    results : PanelResults or GMMResults
        Fitted model results
    filepath : str or Path
        Path to save the JSON file
    """
    filepath = Path(filepath)
    filepath.parent.mkdir(parents=True, exist_ok=True)

    # Build model card
    card = {
        "model_card_version": "1.0",
        "export_timestamp": datetime.now().isoformat(),
    }

    # Model identification
    if isinstance(results, GMMResults):
        card["model_type"] = f"GMM ({results.model_type})"
        card["dep_var"] = results.dep_var
        card["exog_vars"] = results.exog_vars
    elif isinstance(results, PanelResults):
        card["model_type"] = results.model_type
        card["formula"] = results.formula
    else:
        card["model_type"] = type(results).__name__

    # Coefficients
    card["coefficients"] = {
        name: {
            "value": float(coef),
            "std_error": float(results.std_errors[name]),
            "p_value": float(results.pvalues[name]),
        }
        for name, coef in results.params.items()
    }

    # Training info
    if isinstance(results, GMMResults):
        card["training_info"] = {
            "n_observations": int(results.nobs),
            "n_groups": int(results.n_groups),
            "n_instruments": int(results.n_instruments),
            "instrument_ratio": float(results.instrument_ratio),
        }
    elif isinstance(results, PanelResults):
        card["training_info"] = {
            "n_observations": int(results.nobs),
            "n_entities": int(results.n_entities),
            "n_periods": int(results.n_periods) if results.n_periods else None,
        }

    # Diagnostics
    if isinstance(results, GMMResults):
        card["diagnostics"] = {
            "hansen_j": {
                "statistic": float(results.hansen_j.statistic),
                "p_value": float(results.hansen_j.pvalue),
                "conclusion": results.hansen_j.conclusion,
            },
            "ar2_test": {
                "statistic": float(results.ar2_test.statistic),
                "p_value": float(results.ar2_test.pvalue),
                "conclusion": results.ar2_test.conclusion,
            },
        }
    elif isinstance(results, PanelResults):
        card["diagnostics"] = {
            "r_squared": float(results.rsquared) if not np.isnan(results.rsquared) else None,
            "r_squared_adj": float(results.rsquared_adj)
            if not np.isnan(results.rsquared_adj)
            else None,
        }

    # Save
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(card, f, indent=2, ensure_ascii=False)

    return card


print("Function defined.")

Function defined.

In [14]:
# Test with PooledOLS
card_ols = export_model_card(results_ols, MODELS_DIR / "model_card_ols.json")

print("=== OLS Model Card ===")
print(json.dumps(card_ols, indent=2))

=== OLS Model Card ===
{
  "model_card_version": "1.0",
  "export_timestamp": "2026-02-22T15:30:46.268966",
  "model_type": "Pooled OLS",
  "formula": "investment ~ value + capital + sales",
  "coefficients": {
    "Intercept": {
      "value": -0.5074050147613818,
      "std_error": 0.13699516337853299,
      "p_value": 0.00021815049251583396
    },
    "value": {
      "value": 0.2990451388362428,
      "std_error": 0.012882799859334174,
      "p_value": 0.0
    },
    "capital": {
      "value": 0.2772938152703679,
      "std_error": 0.02180354590600603,
      "p_value": 0.0
    },
    "sales": {
      "value": 0.1674205188052195,
      "std_error": 0.010467236041667735,
      "p_value": 0.0
    }
  },
  "training_info": {
    "n_observations": 2000,
    "n_entities": 100,
    "n_periods": 20
  },
  "diagnostics": {
    "r_squared": 0.32730003847491007,
    "r_squared_adj": 0.3262889663884495
  }
}


In [15]:
# Test with GMM
card_gmm = export_model_card(results_gmm, MODELS_DIR / "model_card_gmm.json")

print("=== GMM Model Card ===")
print(json.dumps(card_gmm, indent=2))

=== GMM Model Card ===
{
  "model_card_version": "1.0",
  "export_timestamp": "2026-02-22T15:30:46.290462",
  "model_type": "GMM (difference)",
  "dep_var": "lgd_logit",
  "exog_vars": [
    "saldo_real",
    "pib_growth",
    "selic",
    "collateral_ratio"
  ],
  "coefficients": {
    "L1.lgd_logit": {
      "value": 0.6142366503879944,
      "std_error": 0.018733850921336143,
      "p_value": 0.0
    },
    "saldo_real": {
      "value": 0.10137438567216908,
      "std_error": 0.025837762828285725,
      "p_value": 8.727284243748024e-05
    },
    "pib_growth": {
      "value": 0.050047071901807225,
      "std_error": 0.003040737298738873,
      "p_value": 0.0
    },
    "selic": {
      "value": 0.0013516404435856694,
      "std_error": 0.012879935826867299,
      "p_value": 0.9164221879765064
    },
    "collateral_ratio": {
      "value": 0.19445182537330005,
      "std_error": 0.1481077497088254,
      "p_value": 0.18921400563196533
    }
  },
  "training_info": {
    "n_observa

In [16]:
# Verify files were saved
for name in ["model_card_ols.json", "model_card_gmm.json"]:
    path = MODELS_DIR / name
    size = path.stat().st_size
    print(f"{name}: {size:,} bytes ({size / 1024:.1f} KB)")

print()
print("Model cards exported successfully.")

model_card_ols.json: 891 bytes (0.9 KB)
model_card_gmm.json: 1,352 bytes (1.3 KB)

Model cards exported successfully.


---
*End of solutions for Notebook 02.*